# Load all datapack tables

This loader reads every `.csv`, `.csv.gz`, and `.jsonl.gz` file under `datapack/`. Monthly files for the same table are combined. Access data as `data[folder][table]`.

In [134]:
import sys
import pathlib
import time
import json
import gzip
import numpy as np
import pandas as pd
import os
from pathlib import Path

from IPython.display import display

In [135]:
DATA_ROOT = Path('data/raw')

In [136]:
def _read_table_file(path):
    suffixes = path.suffixes
    if suffixes[-2:] == ['.jsonl', '.gz']:
        return pd.read_json(path, compression='gzip', lines=True)
    try:
        return pd.read_csv(path, compression='infer', encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, compression='infer', encoding='cp1252')
def _table_name(folder, path):
    stem = path.name.removesuffix('.gz').removesuffix('.csv').removesuffix('.jsonl')
    if folder in {'bookings', 'results', 'collections', 'invoices'}:
        return path.relative_to(DATA_ROOT / folder).parts[0] if folder == 'bookings' else folder
    if folder == 'payments':
        return 'gateway_export' if stem.startswith('gateway_export_') else 'settlements'
    if folder == 'campaigns':
        return 'campaign_bookings' if stem.startswith('campaign_bookings') else 'campaigns'
    return stem
def load_datapack(root=DATA_ROOT):
    """Load all CSV/CSV.GZ/JSONL.GZ files, grouped by folder and table."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Datapack folder not found: {root.resolve()}')
    supported = ('*.csv', '*.csv.gz', '*.jsonl.gz')
    paths = sorted(path for pattern in supported for path in root.rglob(pattern))
    grouped_frames = {}
    for path in paths:
        relative_path = path.relative_to(root)
        folder = relative_path.parts[0]
        table = _table_name(folder, path)
        try:
            frame = _read_table_file(path)
        except Exception as exc:
            raise RuntimeError(f'Could not load {relative_path}: {exc}') from exc
        frame['source_file'] = path.name
        frame['source_path'] = relative_path.as_posix()
        grouped_frames.setdefault((folder, table), []).append(frame)
    if not grouped_frames:
        raise FileNotFoundError(f'No supported data files found under {root.resolve()}')
    data = {}
    for (folder, table), frames in grouped_frames.items():
        data.setdefault(folder, {})[table] = pd.concat(frames, ignore_index=True, sort=False)
    return data
data = load_datapack()
# Convenience helpers for inspecting one folder at a time.
bookings_data = data['bookings']
def load_folder(folder):
    return data[folder]
print('Loaded tables:')
for folder, tables in data.items():
    for table, frame in tables.items():
        print(f'{folder}/{table}: {len(frame):,} rows, {len(frame.columns)} columns')


Loaded tables:
bookings/legacy: 1,252,786 rows, 14 columns
bookings/nova: 26,543 rows, 14 columns
campaigns/campaign_bookings: 48,869 rows, 5 columns
campaigns/campaigns: 1 rows, 7 columns
collections/collections: 552,492 rows, 8 columns
invoices/invoices: 470,776 rows, 9 columns
patients/patients: 309,217 rows, 8 columns
payments/gateway_export: 473,507 rows, 9 columns
payments/settlements: 562 rows, 9 columns
plan/plan_2026_H1: 6 rows, 10 columns
reference/cities: 6 rows, 5 columns
reference/clinics: 31 rows, 8 columns
reference/tests_catalogue: 24 rows, 6 columns
results/results: 1,078,420 rows, 7 columns


In [137]:
l = data['bookings']['legacy']

In [138]:
l.info()

<class 'pandas.DataFrame'>
RangeIndex: 1252786 entries, 0 to 1252785
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   booking_id    1252786 non-null  str   
 1   created_at    1252786 non-null  str   
 2   slot_at       1252786 non-null  str   
 3   city          1252786 non-null  str   
 4   clinic_id     1252786 non-null  str   
 5   channel       1252786 non-null  str   
 6   source        1252786 non-null  str   
 7   patient_id    1252786 non-null  str   
 8   patient_name  1252786 non-null  str   
 9   test_code     1251496 non-null  str   
 10  test_price    1252786 non-null  object
 11  home_fee      1252786 non-null  object
 12  source_file   1252786 non-null  str   
 13  source_path   1252786 non-null  str   
dtypes: object(2), str(12)
memory usage: 133.8+ MB


In [139]:
legacy = data['bookings']['legacy'].copy()
nova = data['bookings']['nova'].copy()
n_raw_legacy, n_raw_nova = len(legacy), len(nova)


---
# 1. Cleaning (per source system)

## 1.1 Legacy

**Latin-1 / non-UTF-8 fallback** — ✅ done in the loader (`_read_table_file`): UTF-8 first, then cp1252.

**Remove repeated header rows** — ✅ done, but it does not count them (it removed 21).

In [140]:
# : count the repeated header rows removed above (before - after) and print/log the number
n_before = len(legacy)
legacy = legacy[legacy["city"] != "city"].copy()
n_header_rows = n_before - len(legacy)
print(f"Removed {n_header_rows:,} repeated header rows")


Removed 21 repeated header rows


In [141]:
len(legacy)

1252765

**Parse `dd/mm/yyyy HH:MM` with an explicit format** — ⚠️ partly done. Currently `format="mixed"` + `dayfirst`, not a fixed format; bad dates become NaT but only `created_at` is checked.

In [142]:
# : parse legacy created_at / slot_at with an explicit format ("%d/%m/%Y %H:%M") instead of format="mixed"
# : count NaT in BOTH created_at and slot_at (not only created_at) and report them as rejects


In [143]:
for col in ["created_at", "slot_at"]:
    legacy[col] = pd.to_datetime(legacy[col], format="%d/%m/%Y %H:%M", errors="coerce")

n_bad_dates = legacy[["created_at", "slot_at"]].isna().sum()
print(n_bad_dates)


created_at    0
slot_at       0
dtype: int64


In [144]:
# bookings_all["created_at"].unique()

In [145]:
# bookings_all["created_at"] = pd.to_datetime(
#     bookings_all["created_at"],
#     format="mixed",
#     dayfirst=True,
#     errors="coerce",
#     utc=True,
# )

In [146]:
# bookings_all["created_at"].isna().sum()

In [147]:
# bookings_all["created_at"].count()

In [148]:

# bookings_all["slot_at"] = pd.to_datetime(
#     bookings_all["slot_at"],
#     format="mixed",
#     dayfirst=True,
#     errors="coerce",
#     utc=True,
# )

**Strip whitespace from `booking_id`** — ❌ missing

In [149]:
# : legacy booking_id -> .str.strip()
legacy["booking_id"] = legacy["booking_id"].str.strip()


**Dtypes** — ❌ missing (`test_price` and `home_fee` are still `object`)

In [150]:
# : legacy dtypes
#   city, clinic_id, channel -> category
#   test_price, home_fee   -> Int32 (pd.to_numeric(..., errors="coerce").astype("Int32"))
for col in ["city", "clinic_id", "channel"]:
    legacy[col] = legacy[col].astype("category")
for col in ["test_price", "home_fee"]:
    legacy[col] = pd.to_numeric(legacy[col], errors="coerce").astype("Int32")

print(legacy[["booking_id", "created_at", "slot_at", "city", "clinic_id", "channel", "test_price", "home_fee"]].dtypes)


booking_id               str
created_at    datetime64[us]
slot_at       datetime64[us]
city                category
clinic_id           category
channel             category
test_price             Int32
home_fee               Int32
dtype: object


## 1.2 Nova

**Flatten tests to one row per booking-test** — ✅ done

In [151]:
# Make nova's nested patient and test fields into columns and rows
nova["patient_id"] = nova["patient"].map(
    lambda x: x.get("patient_id") if isinstance(x, dict) else pd.NA
)
nova = nova.explode("tests", ignore_index=True)
nova["test_code"] = nova["tests"].map(
    lambda x: x.get("code") if isinstance(x, dict) else pd.NA
)
nova["test_price"] = nova["tests"].map(
    lambda x: x.get("price") if isinstance(x, dict) else pd.NA
)
print(f"Nova: {n_raw_nova:,} bookings -> {len(nova):,} booking-test rows")


Nova: 26,543 bookings -> 62,439 booking-test rows


**Lowercase `channel` and `source`** — ❌ missing

In [152]:
# : nova channel, source -> .str.strip().str.lower()
for col in ["channel", "source"]:
    nova[col] = nova[col].str.strip().str.lower()



**UTC -> IST, drop the timezone** — ❌ done wrongly. Everything is currently converted to UTC and legacy is not handled. Nova timestamps are UTC (convert to Asia/Kolkata, then drop tz); legacy timestamps are already IST local time (only make them tz-naive).

In [153]:
for col in ["created_at", "slot_at"]:
    nova[col] = (
        pd.to_datetime(nova[col], utc=True)
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
    )
print(nova[["created_at", "slot_at"]].dtypes)
print(nova[["created_at", "slot_at"]].head(3))


created_at    datetime64[us]
slot_at       datetime64[us]
dtype: object
           created_at             slot_at
0 2026-04-21 01:57:00 2026-04-25 10:16:00
1 2026-03-28 03:02:00 2026-04-01 09:17:00
2 2026-03-28 03:02:00 2026-04-01 09:17:00


In [154]:
# bookings_all["created_at"] = bookings_all["created_at"].dt.tz_convert("Asia/Kolkata")

In [155]:
common_columns = [
    "booking_id", "created_at", "slot_at", "city", "clinic_id",
    "channel", "patient_id", "test_code", "test_price", "home_fee",
    "rescheduled_from", "status_at_export", "source_file", "source_path", "source"
]
legacy["booking_system"] = "legacy"
nova["booking_system"] = "nova"

bookings_all = pd.concat(
    [
        legacy.reindex(columns=common_columns + ["booking_system"]),
        nova.reindex(columns=common_columns + ["booking_system"]),
    ],
    ignore_index=True,
)

# Baseline for the reconciliation assert in phase F
n_raw_rows = len(bookings_all)
print(f"Raw booking-test rows: {n_raw_rows:,}")
print(bookings_all.dtypes)


Raw booking-test rows: 1,315,204
booking_id                     str
created_at          datetime64[us]
slot_at             datetime64[us]
city                           str
clinic_id                      str
channel                        str
patient_id                     str
test_code                      str
test_price                  object
home_fee                     Int64
rescheduled_from            object
status_at_export            object
source_file                    str
source_path                    str
source                         str
booking_system                 str
dtype: object


In [156]:
for col in ["test_price", "home_fee"]:
    bookings_all[col] = pd.to_numeric(bookings_all[col], errors="coerce").astype("Int32")


In [157]:
bookings_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 1315204 entries, 0 to 1315203
Data columns (total 16 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   booking_id        1315204 non-null  str           
 1   created_at        1315204 non-null  datetime64[us]
 2   slot_at           1315204 non-null  datetime64[us]
 3   city              1315204 non-null  str           
 4   clinic_id         1315204 non-null  str           
 5   channel           1315204 non-null  str           
 6   patient_id        1315204 non-null  str           
 7   test_code         1313893 non-null  str           
 8   test_price        1315183 non-null  Int32         
 9   home_fee          1315204 non-null  Int32         
 10  rescheduled_from  3542 non-null     object        
 11  status_at_export  62439 non-null    object        
 12  source_file       1315204 non-null  str           
 13  source_path       1315204 non-null  str           
 1

---
# 2. Validation (normalise or reject)
Nothing is silently fixed: every row is either normalised or written to `rejects_bookings.csv` with a `reason`.

## 2.1 Normalise

**Strip whitespace** — ❌ missing

In [158]:
# : strip leading/trailing whitespace on all string columns (booking_id, city, clinic_id, channel, source, test_code, patient_id)
str_cols = ["booking_id", "city", "clinic_id", "channel", "source", "test_code", "patient_id"]
for col in str_cols:
    bookings_all[col] = bookings_all[col].str.strip()


**Lowercase `channel` and `source`** — ❌ missing

In [159]:
# : channel, source -> .str.lower()
for col in ["channel", "source"]:
    bookings_all[col] = bookings_all[col].str.lower()


**Bangalore -> Bengaluru** — ❌ missing

In [160]:
# : city -> replace "Bangalore" with "Bengaluru"
n_bangalore = bookings_all["city"].eq("Bangalore").sum()
bookings_all["city"] = bookings_all["city"].replace({"Bangalore": "Bengaluru"})
print(f"Renamed {n_bangalore:,} Bangalore rows to Bengaluru")
print(sorted(bookings_all["city"].unique()))


Renamed 193,558 Bangalore rows to Bengaluru
['Ahmedabad', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur', 'Pune']


In [161]:
bookings_all["channel"].unique()

<StringArray>
['clinic', 'home']
Length: 2, dtype: str

In [162]:
bookings_all["source"].unique()

<StringArray>
['walkin', 'app', 'phone']
Length: 3, dtype: str

**Uppercase test codes** — ❌ missing

In [163]:
# : test_code -> .str.upper()
bookings_all["test_code"] = bookings_all["test_code"].str.upper()

In [164]:
bookings_all.shape

(1315204, 16)

## 2.2 Reject

In [165]:
# : set up a rejects collection, e.g.
#   rejects = []   # list of DataFrames, each with a `reason` column
#   def reject(df, mask, reason): rejects.append(df.loc[mask].assign(reason=reason)); return df.loc[~mask]
rejects = []

def reject(df, mask, reason):
    """Move rows where mask is True to `rejects` with a reason; return the rest."""
    mask = mask.fillna(True)  # a null in the checked field counts as a failure
    rejects.append(df.loc[mask].assign(reason=reason))
    return df.loc[~mask]


In [166]:
rejects

[]

**Booking ID doesn't match the legacy or Nova pattern** — ❌ missing

In [167]:
# : reject booking_id not matching the legacy regex or the Nova regex (check the data dictionary for both patterns)
LEGACY_ID = r"^BK-[A-Z]{3}-\d{6}-\d{6}$"
NOVA_ID = r"^nova_[a-z]{3}_\d{7}$"

valid_id = (
    (bookings_all["booking_system"].eq("legacy") & bookings_all["booking_id"].str.match(LEGACY_ID))
    | (bookings_all["booking_system"].eq("nova") & bookings_all["booking_id"].str.match(NOVA_ID))
)
bookings_all = reject(bookings_all, ~valid_id, "booking_id pattern")

print(f"Rejected {len(rejects[-1]):,} rows, {rejects[-1]['booking_id'].nunique():,} bookings")
print(rejects[-1]["booking_system"].value_counts())
print(f"Remaining: {len(bookings_all):,}")


Rejected 1,839 rows, 793 bookings
booking_system
legacy    1839
Name: count, dtype: int64
Remaining: 1,313,365


**Test code isn't `T###` or `PK##`** — ❌ missing. The 1,290 null `test_code` rows would also be rejected here.

In [168]:
# : reject test_code not matching r"^(T\d{3}|PK\d{2})$" (this also catches null test_code)
no_tests = bookings_all["test_code"].isna() & bookings_all["test_price"].isna()
bookings_all = reject(bookings_all, no_tests, "no tests")
print(f"no tests: {len(rejects[-1]):,}")


no tests: 21


In [169]:
valid_code = bookings_all["test_code"].str.match(r"^(T\d{3}|PK\d{2})$")
bookings_all = reject(bookings_all, ~valid_code, "test_code pattern")
print(f"test_code pattern: {len(rejects[-1]):,}")


test_code pattern: 1,288


In [170]:
bookings_all["test_code"].nunique()

24

**Negative price** — ❌ missing

In [171]:
# : reject test_price < 0
bookings_all = reject(bookings_all, bookings_all["test_price"].lt(0), "negative test_price")
print(f"negative test_price: {len(rejects[-1]):,}")


negative test_price: 0


**`slot_at` before `created_at`** — ❌ missing

In [172]:
# : reject slot_at < created_at
bookings_all = reject(bookings_all, bookings_all["slot_at"] < bookings_all["created_at"], "slot_at before created_at")
print(f"slot_at before created_at: {len(rejects[-1]):,}")


slot_at before created_at: 2,438


In [173]:
bad_set = ~bookings_all["channel"].isin(["clinic", "home"]) | ~bookings_all["source"].isin(["app", "phone", "walkin"])
bookings_all = reject(bookings_all, bad_set, "channel/source not allowed")
print(f"channel/source: {len(rejects[-1]):,}")
print(f"Remaining: {len(bookings_all):,}")


channel/source: 0
Remaining: 1,309,618


**Booking with no tests** — ⚠️ partly done. The 21 rows with no price are dropped below, but silently with no reason recorded. This breaks "nothing is silently fixed".

In [174]:
bookings_all[bookings_all["test_price"].isna()]

,booking_id,created_at,slot_at,city,clinic_id,channel,patient_id,test_code,test_price,home_fee,rescheduled_from,status_at_export,source_file,source_path,source,booking_system


In [175]:
bookings_all[bookings_all["test_price"].isna()].count()

booking_id          0
created_at          0
slot_at             0
city                0
clinic_id           0
channel             0
patient_id          0
test_code           0
test_price          0
home_fee            0
rescheduled_from    0
status_at_export    0
source_file         0
source_path         0
source              0
booking_system      0
dtype: int64

#### Found 21 rows with 0 test prices and code so we dropped them

In [176]:
bookings_all = bookings_all.loc[bookings_all["test_price"].notna()].copy()

print(bookings_all[["test_code", "test_price"]].isna().sum())
print(bookings_all["test_price"].dtype)

test_code     0
test_price    0
dtype: int64
Int32


**Channel or source outside the allowed set** — ❌ missing

In [177]:
# : reject channel / source not in the allowed sets (see data dictionary)
accepted = bookings_all
assert n_raw_rows == len(accepted) + len(rejects_df), (
    n_raw_rows, len(accepted), len(rejects_df)
)
print(f"raw {n_raw_rows:,} = accepted {len(accepted):,} + rejected {len(rejects_df):,}")


raw 1,315,204 = accepted 1,309,618 + rejected 5,586


## 2.3 Output and reconciliation

**Write `rejects_bookings.csv` with a `reason` column** — ❌ missing

In [178]:
# : pd.concat(rejects).to_csv("rejects_bookings.csv", index=False)


In [179]:
bookings_all["test_code"].value_counts()

test_code
T001    161215
T002    130018
T004     96305
T003     87436
T007     87160
T010     78029
T005     68841
PK01     65927
T006     59831
T008     59523
T009     49540
PK02     45196
PK03     44999
T013     40347
T014     40188
T011     30432
T015     30409
T016     30289
T017     30159
PK04     22689
T012     20295
T019     10325
T018     10260
T020     10205
Name: count, dtype: int64

In [180]:
bookings_all["has_package_row"] = bookings_all["test_code"].str.startswith("PK")

grp = bookings_all.groupby("booking_id", sort=False)
bookings = grp.agg(
    n_tests=("test_code", "count"),
    price_sum=("test_price", "sum"),
    home_fee=("home_fee", "first"),
    has_package=("has_package_row", "any"),
    created_at=("created_at", "first"),
    slot_at=("slot_at", "first"),
    city=("city", "first"),
    clinic_id=("clinic_id", "first"),
    channel=("channel", "first"),
    source=("source", "first"),
    patient_id=("patient_id", "first"),
    rescheduled_from=("rescheduled_from", "first"),
    status_at_export=("status_at_export", "first"),
    booking_system=("booking_system", "first"),
).reset_index()

bookings["amount_rs"] = (bookings["price_sum"] + bookings["home_fee"]).astype("int64")
bookings = bookings.drop(columns="price_sum")
bookings["month"] = bookings["slot_at"].dt.to_period("M").astype(str)
bookings["is_reschedule"] = bookings["rescheduled_from"].notna()

print(f"{len(bookings):,} bookings, {int(bookings['n_tests'].sum()):,} tests")


566,002 bookings, 1,309,618 tests


In [181]:
bookings

,booking_id,n_tests,home_fee,has_package,created_at,slot_at,city,clinic_id,channel,source,patient_id,rescheduled_from,status_at_export,booking_system,amount_rs,month,is_reschedule
0,BK-AHM-202501-000001,3,0,False,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,None,None,legacy,2000,2025-01,False
1,BK-AHM-202501-000002,1,0,False,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,None,None,legacy,650,2025-01,False
2,BK-AHM-202501-000003,4,150,False,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,None,None,legacy,1820,2025-01,False
3,BK-AHM-202501-000004,1,0,False,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,PAHM0007069,None,None,legacy,160,2025-01,False
4,BK-AHM-202501-000005,2,0,False,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,PAHM0002520,None,None,legacy,1900,2025-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565997,nova_jai_0055749,2,150,False,2026-06-02 04:49:00,2026-06-08 11:54:00,Jaipur,JAI-02,home,app,PJAI0028824,nova_jai_0054922,booked,nova,1150,2026-06,True
565998,nova_jai_0055750,3,150,False,2026-06-15 23:28:00,2026-06-18 08:22:00,Jaipur,JAI-01,home,walkin,PJAI0028764,nova_jai_0054877,booked,nova,1950,2026-06,True
565999,nova_jai_0055751,1,0,True,2026-06-19 05:36:00,2026-06-26 11:47:00,Jaipur,JAI-02,clinic,walkin,PJAI0028485,nova_jai_0052936,booked,nova,1499,2026-06,True
566000,nova_jai_0055752,3,150,False,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,nova_jai_0052790,booked,nova,1470,2026-06,True


In [182]:
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
current_bookings = bookings.loc[~is_superseded].copy()

print(f"Removed {is_superseded.sum():,} superseded Nova bookings")
print(f"Kept {len(current_bookings):,} bookings, {int(current_bookings['n_tests'].sum()):,} tests")
print(f"Reschedule replacements: {current_bookings['is_reschedule'].sum():,}")


Removed 1,499 superseded Nova bookings
Kept 564,503 bookings, 1,306,077 tests
Reschedule replacements: 1,500


In [183]:
bookings['patient_id'].count()

np.int64(566002)

## Loading Collections

In [184]:
coll = data["collections"]["collections"].copy()

In [185]:
coll["status"].value_counts()

status
collected    473320
no_show       79172
Name: count, dtype: int64

In [186]:
coll["booking_id"] = coll["booking_id"].str.strip()

In [187]:
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")

In [188]:
coll[["booking_id", "status", "collected_at"]].value_counts()

booking_id            status     collected_at       
BK-AHM-202501-000032  collected  2025-01-02 12:22:00    2
BK-AHM-202501-000102  collected  2025-01-30 13:51:00    2
BK-AHM-202501-001036  collected  2025-01-11 13:47:00    2
BK-AHM-202501-001426  collected  2025-01-26 18:55:00    2
BK-AHM-202501-001432  collected  2025-01-08 07:14:00    2
                                                       ..
BK-PUN-202606-079974  collected  2026-06-26 12:09:00    1
BK-PUN-202606-079975  collected  2026-06-17 13:20:00    1
BK-PUN-202606-079977  collected  2026-06-13 10:40:00    1
BK-PUN-202606-079978  collected  2026-06-28 12:04:00    1
BK-PUN-202606-079979  collected  2026-06-17 09:23:00    1
Name: count, Length: 470426, dtype: int64

In [189]:
coll[coll["booking_id"]=="BK-AHM-202501-000032"]

,booking_id,clinic_id,channel,status,collected_at,phlebotomist_id,source_file,source_path
30,BK-AHM-202501-000032,AHM-04,clinic,collected,2025-01-02 12:22:00,NaN,collections_AHM_2025-01.csv.gz,collections/collections_AHM_2025-01.csv.gz
5446,BK-AHM-202501-000032,AHM-04,clinic,collected,2025-01-02 12:22:00,NaN,collections_AHM_2025-01.csv.gz,collections/collections_AHM_2025-01.csv.gz


In [190]:
n_coll_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")

In [191]:
coll_clean = coll.loc[~dups].copy()

In [192]:


print(f"Collections: {n_coll_in:,} in, {dups.sum():,} double-scans removed, {len(coll_clean):,} kept")
assert n_coll_in == len(coll_clean) + dups.sum()
assert coll_clean["booking_id"].is_unique

Collections: 552,492 in, 2,894 double-scans removed, 549,598 kept


**Left-join collections** (`one_to_one`, no row fan-out) — ❌ missing

In [193]:
n_before = len(current_bookings)
joined = current_bookings.merge(
    coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]],
    on="booking_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert len(joined) == n_before, "join fanned out"
print(joined["_merge"].value_counts())


_merge
both          546207
left_only      18296
right_only         0
Name: count, dtype: int64


In [194]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
print(joined["status"].value_counts())


status
collected               467504
no_show                  78703
cancelled_or_missing     18296
Name: count, dtype: int64


In [195]:
joined

,booking_id,n_tests,home_fee,has_package,created_at,slot_at,city,clinic_id,channel,source,...,rescheduled_from,status_at_export,booking_system,amount_rs,month,is_reschedule,status,collected_at,phlebotomist_id,_merge
0,BK-AHM-202501-000001,3,0,False,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,...,None,None,legacy,2000,2025-01,False,collected,2025-01-09 09:09:00,NaN,both
1,BK-AHM-202501-000002,1,0,False,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,...,None,None,legacy,650,2025-01,False,collected,2025-01-06 13:44:00,NaN,both
2,BK-AHM-202501-000003,4,150,False,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,...,None,None,legacy,1820,2025-01,False,collected,2025-01-28 13:37:00,PHAHM047,both
3,BK-AHM-202501-000004,1,0,False,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,...,None,None,legacy,160,2025-01,False,cancelled_or_missing,NaT,NaN,left_only
4,BK-AHM-202501-000005,2,0,False,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,...,None,None,legacy,1900,2025-01,False,collected,2025-01-11 09:59:00,NaN,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564498,nova_jai_0055749,2,150,False,2026-06-02 04:49:00,2026-06-08 11:54:00,Jaipur,JAI-02,home,app,...,nova_jai_0054922,booked,nova,1150,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
564499,nova_jai_0055750,3,150,False,2026-06-15 23:28:00,2026-06-18 08:22:00,Jaipur,JAI-01,home,walkin,...,nova_jai_0054877,booked,nova,1950,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
564500,nova_jai_0055751,1,0,True,2026-06-19 05:36:00,2026-06-26 11:47:00,Jaipur,JAI-02,clinic,walkin,...,nova_jai_0052936,booked,nova,1499,2026-06,True,cancelled_or_missing,NaT,NaN,left_only
564501,nova_jai_0055752,3,150,False,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,...,nova_jai_0052790,booked,nova,1470,2026-06,True,cancelled_or_missing,NaT,NaN,left_only


In [196]:
skew = (joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)).fillna(False)

In [197]:
skew

0         False
1         False
2         False
3         False
4         False
          ...  
564498    False
564499    False
564500    False
564501    False
564502    False
Length: 564503, dtype: bool

In [109]:
skew = (joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)).fillna(False)
joined["flag_clock_skew"] = skew
joined.loc[skew, "collected_at"] = pd.NaT
bookings_full = joined.drop(columns="_merge")
print(f"Clock skew: {int(skew.sum()):,}")
print(f"collected_at null: {bookings_full['collected_at'].isna().sum():,}")


Clock skew: 938
collected_at null: 97,937


In [ ]:
# : bookings.merge(collections, on="booking_id", how="left", validate="one_to_one")
# assert len(result) == len(bookings)


**Fill missing collection status with `cancelled_or_missing`** — ❌ missing

In [ ]:
# : collection_status = collection_status.fillna("cancelled_or_missing")


**Blank `collected_at` and set `flag_clock_skew`** when the collection is more than a day before the slot — ❌ missing

In [ ]:
# : flag_clock_skew = collected_at < slot_at - pd.Timedelta(days=1)
# : set collected_at = NaT where flag_clock_skew is True


**Map duplicate patient IDs to one `person_id`** (phone + name + DOB) and re-key bookings — ❌ missing

In [ ]:
# : build person_id from (phone, name, DOB) using the patients table; map every patient_id -> person_id
# : replace patient_id in bookings with person_id
